In [9]:
import time
from PIL import Image
import torch
from torchvision.transforms.functional import to_pil_image


import time
import urllib3
from PIL import Image
# from src_depth.model import get_model
# from src_depth.preprocess import make_depth_transform, render_depth
# from src_depth.inference import infer_depth

In [ ]:
def infer_depth(model, transform, image: Image):
    scale_factor = 1
    rescaled_image = image.resize(
        (scale_factor * image.width, scale_factor * image.height)
    )
    transformed_image = transform(rescaled_image)
    batch = transformed_image.unsqueeze(0).cuda()  # Make a batch of one image

    a = time.time()
    with torch.inference_mode():
        result = model.whole_inference(batch, img_meta=None, rescale=True)
    print(f"Total time inference: {time.time() - a}")
    values = result.squeeze().cpu()

    min_value, max_value = values.min(), values.max()
    normalized_values = (values - min_value) / (max_value - min_value)
    return to_pil_image(normalized_values)

Using cache found in /home/ezalos/.cache/torch/hub/facebookresearch_dinov2_main
/home/ezalos/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/ezalos/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/ezalos/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_linear4_head.pth" to /home/ezalos/.cache/torch/hub/checkpoints/dinov2_vits14_linear4_head.pth
100%|██████████| 7.33M/7.33M [00:00<00:00, 86.3MB/s]


In [20]:
from io import BytesIO
import urllib3
from PIL import Image
import numpy as np

def load_image_from_url(url: str) -> Image:
    http = urllib3.PoolManager()
    response = http.request("GET", url)
    return Image.open(BytesIO(response.data)).convert("RGB")


EXAMPLE_IMAGE_URL = "https://dl.fbaipublicfiles.com/dinov2/images/example.jpg"
image = load_image_from_url(EXAMPLE_IMAGE_URL)
image.save("./data/img_0.png")

import torch

# DINOv2
dinov2_vits14_lc = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14_lc")
# dinov2_vitb14_lc = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_lc")
# dinov2_vitl14_lc = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14_lc")
# dinov2_vitg14_lc = torch.hub.load("facebookresearch/dinov2", "dinov2_vitg14_lc")

# DINOv2 with registers
dinov2_vits14_reg_lc = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14_reg_lc")
# dinov2_vitb14_reg_lc = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_reg_lc")
# dinov2_vitl14_reg_lc = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14_reg_lc")
# dinov2_vitg14_reg_lc = torch.hub.load("facebookresearch/dinov2", "dinov2_vitg14_reg_lc")

with torch.inference_mode():
    image = image.resize((224, 224))
    t_image = torch.tensor(np.array(image)).to(torch.float32)
    # t_image = t_image.permute(2, 0, 1).unsqueeze(0)  # Convert to [B,C,H,W] format
    # out_normal = dinov2_vits14_lc.forward(t_image)
    # out_reg = dinov2_vits14_reg_lc.forward(t_image)

    our_reg = dinov2_vits14_reg_lc.whole_inference(t_image, img_meta=None, rescale=True)

print(f"{out_reg.shape = }")
print(f"{out_normal.shape = }")
# transform = make_depth_transform()
# model = get_model()

# result = infer_depth(model, transform, image)
# depth_image = render_depth(result.squeeze().cpu())
# depth_image.save("img_0d.png")

Using cache found in /home/ezalos/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /home/ezalos/.cache/torch/hub/facebookresearch_dinov2_main


AttributeError: '_LinearClassifierWrapper' object has no attribute 'whole_inference'